In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras.src.losses import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
def ensure_dataframe(data: pd.DataFrame | np.ndarray, column_names: list[str]) -> pd.DataFrame:
    if isinstance(data, pd.DataFrame):
        return data[column_names]
    else:
        return pd.DataFrame(data, columns=column_names)


def draw_boxplot(data: pd.DataFrame | np.ndarray, column_names: list[str], title: str) -> None:
    df = ensure_dataframe(data, column_names)
    mean_value = df.mean().mean()

    plt.figure(figsize=(15, 10))
    sns.boxplot(data=df, palette="coolwarm")

    plt.axhline(y=mean_value, color='red', linestyle='--', label='Mean')
    plt.ylabel(ylabel="Value")
    plt.xlabel(xlabel="Feature")

    plt.title(label=title)
    plt.legend()
    plt.show()


def draw_corr_matrix(data, column_names, title):
    df = ensure_dataframe(data, column_names)
    corr_matrix = df.corr().abs()

    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")

    plt.title(label=title)
    plt.show()

In [ ]:
chunk_size = 2 * 10 ** 4
corr_threshold = 0.9

work_dir = Path.cwd().absolute()
raw_train_path = work_dir / "data" / "raw" / "train.csv"
raw_test_path = work_dir / "data" / "raw" / "test.csv"

split_train_path = work_dir / "data" / "split" / "train_split.csv"
split_val_path = work_dir / "data" / "split" / "val_split.csv"

processed_train_path = work_dir / "data" / "processed" / "train_processed.npz"
processed_val_path = work_dir / "data" / "processed" / "val_processed.npz"

In [ ]:
is_first = True
chunk_iterator = pd.read_csv(raw_train_path, chunksize=chunk_size)
for chunk in chunk_iterator:
    train_chunk, val_chunk = train_test_split(chunk, test_size=0.2)

    train_chunk.to_csv(split_train_path, mode="a", index=False, header=is_first)
    val_chunk.to_csv(split_val_path, mode="a", index=False, header=is_first)
    is_first = False

In [ ]:
train_df = pd.read_csv(split_train_path)
val_df = pd.read_csv(split_val_path)

In [ ]:
train_df.head()

In [ ]:
train_df.tail()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.shape

In [ ]:
val_df.head()

In [ ]:
val_df.tail()

In [ ]:
val_df.info()

In [ ]:
val_df.describe()

In [ ]:
val_df.shape

In [ ]:
print(train_df.isna().sum())
print(val_df.isna().sum())

In [ ]:
print(train_df.duplicated().sum())
print(train_df.duplicated().sum())

In [ ]:
cat_cols = [col for col in train_df.columns if col.startswith("cat")]
num_cols = [col for col in train_df.columns if col.startswith("cont")]

In [ ]:
draw_corr_matrix(train_df, num_cols, "Correlation matrix")

corr_matrix = train_df[num_cols].corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > corr_threshold)]

print(f"Will be dropped because of high correlation: {to_drop}")

In [ ]:
train_df = train_df.drop(columns=to_drop)
val_df = val_df.drop(columns=to_drop)

num_cols = [col for col in num_cols if col not in to_drop]

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
x_train_cat_encoded = encoder.fit_transform(train_df[cat_cols])
x_val_cat_encoded = encoder.transform(val_df[cat_cols])

print(train_df[cat_cols].shape)
print(x_train_cat_encoded.shape)

In [ ]:
scaler = StandardScaler()
x_train_num_scaled = scaler.fit_transform(train_df[num_cols])
x_val_num_scaled = scaler.transform(val_df[num_cols])

draw_boxplot(train_df, num_cols, "Original train DataFrame")
draw_boxplot(x_train_num_scaled, num_cols, "Scaled train DataFrame")

draw_boxplot(val_df, num_cols, "Original validation DataFrame")
draw_boxplot(x_val_num_scaled, num_cols, "Scaled validation DataFrame")

In [ ]:
y_train_split = train_df["loss"].values
y_val_split = val_df["loss"].values

x_train_ready = np.column_stack([x_train_cat_encoded, x_train_num_scaled])
x_val_ready = np.column_stack([x_val_cat_encoded, x_val_num_scaled])

np.savez_compressed(processed_train_path, X=x_train_ready, Y=y_train_split)
np.savez_compressed(processed_val_path, X=x_val_ready, Y=y_val_split)

In [ ]:
import jax
devices = jax.devices()
print(devices)

backend_platform = jax.default_backend()

if backend_platform.lower() == "gpu":
    print("JAX GPU backend")
else:
    print("JAX CPU backend")

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'jax'

from keras.models import Sequential
from keras.layers import Dense, Dropout, Input

input_share_dim = (x_train_ready.shape[1],)

model = Sequential([
    Input(shape=input_share_dim),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(1, activation="linear"),
])

model.compile(optimizer="adam", loss="huber", metrics=["mae"])
model.summary()

In [ ]:
history = model.fit(
    x_train_ready, y_train_split,
    epochs=100,
    batch_size=512,
    validation_data=(x_val_ready, y_val_split),
    verbose=1,
)

In [ ]:
y_train_log = np.log1p(y_train_split)
y_val_log = np.log1p(y_val_split)

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"

input_share_dim = (x_train_ready.shape[1],)

model_log = Sequential([
    Input(shape=input_share_dim),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="linear"),
])

model_log.compile(optimizer="adam", loss="mae", metrics=["mae"])
model_log.summary()

In [ ]:
history_log = model_log.fit(
    x_train_ready, y_train_log,
    epochs=100,
    batch_size=512,
    validation_data=(x_val_ready, y_val_log),
    verbose=1,
)


In [ ]:
y_pred_log = model_log.predict(x_val_ready).flatten()
y_pred = np.expm1(y_pred_log)

final_mae = mean_absolute_error(y_val_split, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_val_split, y_pred))

print(f"MAE: {final_mae}, RMSE: {final_rmse}")